# MNIST with the EGGROLL estimator

This notebook trains a linear MNIST classifier using the rank-`r` two-point EGGROLL estimator

$$\hat g(W) = \frac{1}{r}\sum_{i=1}^r \frac{f(W + \sigma a_i b_i^\top) - f(W - \sigma a_i b_i^\top)}{2\sigma} a_i b_i^\top,$$

where `a_i ~ N(0, I_784)` and `b_i ~ N(0, I_10)`. The rank `r` stays fixed, and each training iteration samples fresh Gaussian directions.

In [1]:
# Run this if torch/torchvision are missing in your notebook kernel.
%pip install torch torchvision tqdm matplotlib

In [3]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

seed = 0
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='cuda')

In [4]:
data_dir = Path("data")
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_data = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
test_data = datasets.MNIST(data_dir, train=False, download=True, transform=transform)

# EGGROLL is expensive because each step uses 2r loss evaluations. Start small,
# then increase this to len(train_data) once the run looks healthy.
train_limit = 10_000
train_subset = Subset(train_data, range(train_limit))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

input_dim = 28 * 28
num_classes = 10
len(train_subset), len(test_data)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 509kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.67MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.5MB/s]


(10000, 10000)

In [5]:
def loss_at(W, X, y):
    logits = X @ W
    return F.cross_entropy(logits, y)


@torch.no_grad()
def eggroll_estimator(W, X, y, *, sigma, rank):
    """Rank-r EGGROLL estimator with fresh Gaussian directions every call."""
    grad_hat = torch.zeros_like(W)

    for _ in range(rank):
        a = torch.randn(W.shape[0], device=W.device)
        b = torch.randn(W.shape[1], device=W.device)
        direction = torch.outer(a, b)

        f_plus = loss_at(W + sigma * direction, X, y)
        f_minus = loss_at(W - sigma * direction, X, y)
        directional_derivative = (f_plus - f_minus) / (2.0 * sigma)
        grad_hat.add_(directional_derivative * direction)

    return grad_hat / rank


@torch.no_grad()
def accuracy(W, loader):
    correct = 0
    total = 0
    total_loss = 0.0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)
        logits = X @ W
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return total_loss / total, correct / total

In [6]:
# Hyperparameters. Increase rank for lower variance; decrease learning_rate if loss spikes.
rank = 8
sigma = 1e-3
learning_rate = 5e-2
epochs = 3
grad_clip_norm = 20.0

# Linear softmax classifier parameter W in R^{784 x 10}.
W = 0.01 * torch.randn(input_dim, num_classes, device=device)

history = []

In [7]:
for epoch in range(1, epochs + 1):
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}")

    for step, (X, y) in enumerate(progress, start=1):
        X = X.to(device)
        y = y.to(device)

        grad_hat = eggroll_estimator(W, X, y, sigma=sigma, rank=rank)
        grad_norm = grad_hat.norm()
        if grad_norm > grad_clip_norm:
            grad_hat.mul_(grad_clip_norm / grad_norm)

        W.sub_(learning_rate * grad_hat)

        batch_loss = loss_at(W, X, y).item()
        running_loss += batch_loss
        progress.set_postfix(loss=running_loss / step, grad_norm=grad_norm.item())

    train_loss, train_acc = accuracy(W, train_loader)
    test_loss, test_acc = accuracy(W, test_loader)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })
    print(
        f"epoch {epoch}: train loss={train_loss:.4f}, train acc={train_acc:.3%}, "
        f"test loss={test_loss:.4f}, test acc={test_acc:.3%}"
    )

epoch 1/3:   0%|          | 0/78 [00:00<?, ?it/s]

epoch 1: train loss=1.2090, train acc=62.310%, test loss=1.1731, test acc=62.370%


epoch 2/3:   0%|          | 0/78 [00:00<?, ?it/s]

epoch 2: train loss=1.0712, train acc=68.920%, test loss=1.0664, test acc=68.850%


epoch 3/3:   0%|          | 0/78 [00:00<?, ?it/s]

epoch 3: train loss=1.0719, train acc=71.865%, test loss=1.0793, test acc=71.340%


In [ ]:
history

## Compare rank $r$ and smoothing scale $\sigma$

The sweep below trains every $(r, \sigma)$ configuration for the same number of minibatch updates and plots the resulting MNIST test accuracy. Each optimization step still draws fresh Gaussian vectors $a_i$ and $b_i$.

In [ ]:
@torch.no_grad()
def run_eggroll_sweep(ranks, sigmas, *, steps=40):
    results = []
    initial_W = 0.01 * torch.randn(input_dim, num_classes, device=device)

    for r_value in ranks:
        for sigma_value in sigmas:
            # Every configuration starts from identical model parameters.
            W_trial = initial_W.clone()
            trial_loader = DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=True,
                drop_last=True,
                generator=torch.Generator().manual_seed(seed),
            )

            progress = tqdm(
                zip(range(steps), trial_loader),
                total=steps,
                desc=f"r={r_value}, sigma={sigma_value:g}",
            )
            for _, (X, y) in progress:
                X = X.to(device)
                y = y.to(device)
                grad_hat = eggroll_estimator(
                    W_trial, X, y, sigma=sigma_value, rank=r_value
                )
                grad_norm = grad_hat.norm()
                if grad_norm > grad_clip_norm:
                    grad_hat.mul_(grad_clip_norm / grad_norm)
                W_trial.sub_(learning_rate * grad_hat)

            test_loss, test_acc = accuracy(W_trial, test_loader)
            results.append({
                "rank": r_value,
                "sigma": sigma_value,
                "test_loss": test_loss,
                "test_acc": test_acc,
            })
            print(
                f"r={r_value:>2}, sigma={sigma_value:g}: "
                f"test loss={test_loss:.4f}, test accuracy={test_acc:.2%}"
            )

    return results


ranks_to_compare = [1, 2, 4, 8]
sigmas_to_compare = [1e-4, 1e-3, 1e-2, 1e-1]
sweep_results = run_eggroll_sweep(
    ranks_to_compare, sigmas_to_compare, steps=40
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for r_value in ranks_to_compare:
    rows = [row for row in sweep_results if row["rank"] == r_value]
    ax.plot(
        [row["sigma"] for row in rows],
        [100 * row["test_acc"] for row in rows],
        marker="o",
        linewidth=2,
        label=f"r = {r_value}",
    )

ax.set_xscale("log")
ax.set_xlabel(r"Smoothing scale $\sigma$")
ax.set_ylabel("Test accuracy (%)")
ax.set_title(r"EGGROLL: effect of rank $r$ and smoothing scale $\sigma$")
ax.grid(True, which="both", alpha=0.3)
ax.legend(title="Estimator rank")
fig.tight_layout()
plt.show()

In [8]:
# Show a few predictions from the trained EGGROLL classifier.
X, y = next(iter(test_loader))
X = X.to(device)
with torch.no_grad():
    preds = (X @ W).argmax(dim=1).cpu()

list(zip(preds[:20].tolist(), y[:20].tolist()))

[(7, 7),
 (8, 2),
 (1, 1),
 (0, 0),
 (4, 4),
 (1, 1),
 (6, 4),
 (9, 9),
 (2, 5),
 (9, 9),
 (0, 0),
 (6, 6),
 (9, 9),
 (0, 0),
 (1, 1),
 (3, 5),
 (9, 9),
 (7, 7),
 (3, 3),
 (4, 4)]